# 🌲 Notebook 1 — Data Engineer
## Forestry Image Classification | ISB46703

**Role:** Data Engineer  
**Tasks:**
- Collect images using web crawler
- Standardize the data
- Create and split the dataset (train / val / test)


## 📦 1. Install Dependencies

In [ ]:
# Run this cell once to install required libraries
!pip install icrawler tensorflow keras pillow numpy matplotlib tqdm scikit-learn

## 📚 2. Import Libraries

In [ ]:
import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image, ImageFile
from tqdm import tqdm
from icrawler.builtin import GoogleImageCrawler, BingImageCrawler
import warnings
warnings.filterwarnings('ignore')
ImageFile.LOAD_TRUNCATED_IMAGES = True

print('✅ Libraries imported successfully')

## 🔍 3. Define Dataset Classes & Paths

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────
CLASSES = {
    'healthy_forest'   : 'healthy green forest trees dense canopy',
    'deforested'       : 'deforestation cleared land logging aerial view',
    'forest_fire'      : 'forest fire burning wildfire smoke aerial',
    'flooded_forest'   : 'flooded forest waterlogged trees aerial view',
    'diseased_forest'  : 'diseased dead trees forest blight bark beetle'
}

IMAGES_PER_CLASS = 200   # Set to 2000 for full 10,000-image dataset (takes longer)
BASE_DIR         = './dataset'
RAW_DIR          = './dataset/raw'
TRAIN_SPLIT      = 0.70
VAL_SPLIT        = 0.15
TEST_SPLIT       = 0.15
IMG_SIZE         = (224, 224)

print(f'📂 Classes  : {list(CLASSES.keys())}')
print(f'🖼️  Images   : {IMAGES_PER_CLASS} per class ({IMAGES_PER_CLASS * len(CLASSES)} total)')
print(f'📊 Split    : Train {int(TRAIN_SPLIT*100)}% / Val {int(VAL_SPLIT*100)}% / Test {int(TEST_SPLIT*100)}%')

## 🕷️ 4. Web Crawling — Collect Images

In [ ]:
def crawl_images(class_name, query, num_images, output_dir):
    """Crawl images using Bing Image Crawler."""
    os.makedirs(output_dir, exist_ok=True)
    print(f'\n🔍 Crawling: [{class_name}] — query: "{query}"')
    
    try:
        crawler = BingImageCrawler(
            feeder_threads=2,
            parser_threads=2,
            downloader_threads=4,
            storage={'root_dir': output_dir}
        )
        crawler.crawl(
            keyword=query,
            max_num=num_images,
            min_size=(150, 150),
            file_idx_offset=0
        )
        count = len(os.listdir(output_dir))
        print(f'   ✅ Downloaded {count} images')
        return count
    except Exception as e:
        print(f'   ❌ Error: {e}')
        return 0

# ── RUN CRAWLING ───────────────────────────────────────────────
total_downloaded = 0
for class_name, query in CLASSES.items():
    output_path = os.path.join(RAW_DIR, class_name)
    count = crawl_images(class_name, query, IMAGES_PER_CLASS, output_path)
    total_downloaded += count

print(f'\n📦 Total images downloaded: {total_downloaded}')

## 🧹 5. Data Standardization — Clean & Resize Images

In [ ]:
def standardize_images(raw_dir, class_name, img_size=(224, 224)):
    """Validate, convert to RGB, resize, and remove corrupt images."""
    class_dir = os.path.join(raw_dir, class_name)
    if not os.path.exists(class_dir):
        return 0, 0
    
    files      = os.listdir(class_dir)
    valid      = 0
    removed    = 0
    
    for fname in tqdm(files, desc=f'Standardizing {class_name}'):
        fpath = os.path.join(class_dir, fname)
        try:
            with Image.open(fpath) as img:
                # Convert to RGB (removes alpha channels, handles grayscale)
                img = img.convert('RGB')
                # Resize to standard size
                img = img.resize(img_size, Image.LANCZOS)
                # Save as JPEG
                new_name = f'{class_name}_{valid:04d}.jpg'
                img.save(os.path.join(class_dir, new_name), 'JPEG', quality=90)
                # Remove old file if renamed
                if fname != new_name:
                    os.remove(fpath)
                valid += 1
        except Exception:
            os.remove(fpath)
            removed += 1
    
    print(f'   [{class_name}] ✅ {valid} valid | ❌ {removed} removed')
    return valid, removed

# ── RUN STANDARDIZATION ────────────────────────────────────────
print('\n🧹 Standardizing images...')
summary = {}
for class_name in CLASSES:
    valid, removed = standardize_images(RAW_DIR, class_name, IMG_SIZE)
    summary[class_name] = {'valid': valid, 'removed': removed}

print('\n📋 Summary:')
for cls, stats in summary.items():
    print(f'   {cls:<20}: {stats["valid"]} valid, {stats["removed"]} removed')

## ✂️ 6. Create Dataset Split (Train / Val / Test)

In [ ]:
def create_dataset_splits(raw_dir, base_dir, classes, train=0.7, val=0.15, test=0.15):
    """Split images into train/val/test directories."""
    assert abs(train + val + test - 1.0) < 1e-6, 'Splits must sum to 1.0'
    
    split_counts = {'train': {}, 'val': {}, 'test': {}}
    
    for class_name in classes:
        src_dir = os.path.join(raw_dir, class_name)
        if not os.path.exists(src_dir):
            continue
        
        images = sorted([f for f in os.listdir(src_dir) if f.endswith('.jpg')])
        random.seed(42)
        random.shuffle(images)
        
        n       = len(images)
        n_train = int(n * train)
        n_val   = int(n * val)
        
        splits = {
            'train': images[:n_train],
            'val'  : images[n_train:n_train + n_val],
            'test' : images[n_train + n_val:]
        }
        
        for split_name, split_files in splits.items():
            dst_dir = os.path.join(base_dir, split_name, class_name)
            os.makedirs(dst_dir, exist_ok=True)
            for fname in split_files:
                shutil.copy2(os.path.join(src_dir, fname), os.path.join(dst_dir, fname))
            split_counts[split_name][class_name] = len(split_files)
    
    return split_counts

# ── CREATE SPLITS ──────────────────────────────────────────────
print('\n✂️  Splitting dataset...')
split_counts = create_dataset_splits(RAW_DIR, BASE_DIR, CLASSES, TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT)

print('\n📊 Dataset split results:')
print(f'{"Class":<22} {"Train":>8} {"Val":>8} {"Test":>8} {"Total":>8}')
print('-' * 56)
for cls in CLASSES:
    tr = split_counts['train'].get(cls, 0)
    va = split_counts['val'].get(cls, 0)
    te = split_counts['test'].get(cls, 0)
    print(f'{cls:<22} {tr:>8} {va:>8} {te:>8} {tr+va+te:>8}')
print('-' * 56)
tot_tr = sum(split_counts['train'].values())
tot_va = sum(split_counts['val'].values())
tot_te = sum(split_counts['test'].values())
print(f'{"TOTAL":<22} {tot_tr:>8} {tot_va:>8} {tot_te:>8} {tot_tr+tot_va+tot_te:>8}')

## 📊 7. Visualize Dataset Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('🌲 Forestry Dataset Overview', fontsize=16, fontweight='bold')

# Bar chart — images per class per split
ax1 = axes[0]
x   = np.arange(len(CLASSES))
w   = 0.25
labels = list(CLASSES.keys())
tr_vals = [split_counts['train'].get(c, 0) for c in labels]
va_vals = [split_counts['val'].get(c, 0)   for c in labels]
te_vals = [split_counts['test'].get(c, 0)  for c in labels]

ax1.bar(x - w, tr_vals, w, label='Train', color='#2ECC71')
ax1.bar(x,     va_vals, w, label='Val',   color='#3498DB')
ax1.bar(x + w, te_vals, w, label='Test',  color='#E74C3C')
ax1.set_xticks(x)
ax1.set_xticklabels([l.replace('_', '\n') for l in labels], fontsize=9)
ax1.set_ylabel('Number of Images')
ax1.set_title('Images per Class & Split')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Pie chart — overall split
ax2 = axes[1]
totals = [tot_tr, tot_va, tot_te]
ax2.pie(totals, labels=['Train', 'Validation', 'Test'],
        colors=['#2ECC71', '#3498DB', '#E74C3C'],
        autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax2.set_title(f'Dataset Split (Total: {tot_tr+tot_va+tot_te} images)')

plt.tight_layout()
plt.savefig('./results/dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: results/dataset_distribution.png')

## 🖼️ 8. Preview Sample Images

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 4, figsize=(16, 4 * len(CLASSES)))
fig.suptitle('🌲 Sample Images per Class', fontsize=16, fontweight='bold', y=1.01)

for row, class_name in enumerate(CLASSES):
    class_dir = os.path.join(BASE_DIR, 'train', class_name)
    if not os.path.exists(class_dir):
        continue
    images = [f for f in os.listdir(class_dir) if f.endswith('.jpg')][:4]
    
    for col, fname in enumerate(images):
        ax = axes[row][col]
        img = mpimg.imread(os.path.join(class_dir, fname))
        ax.imshow(img)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(class_name.replace('_', ' ').title(),
                         rotation=90, fontsize=11, fontweight='bold',
                         labelpad=10, va='center')

plt.tight_layout()
plt.savefig('./results/sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Saved: results/sample_images.png')

---
## ✅ Data Engineering Complete!

**Summary:**
- ✅ Images crawled from Bing Image Search
- ✅ All images standardized to 224×224 RGB JPEG
- ✅ Corrupt/invalid images removed
- ✅ Dataset split: 70% train / 15% val / 15% test
- ✅ 5 forestry classes ready for modelling

**Next step:** Run `02_data_scientist.ipynb` to train the CNN models.